In [2]:
!pip install sentence-transformers faiss-cpu rank-bm25 langchain langchain-text-splitters

In [3]:
import os
import glob
import numpy as np
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from rank_bm25 import BM25Okapi

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from transformers import pipeline

Load Data

In [4]:
documents = []
doc_ids = []

DOC_PATH = "/home/jupyter/advanced/nlp/documents/*.txt"

for filepath in glob.glob(DOC_PATH):

    with open(filepath, "r", encoding="utf-8") as f:

        text = f.read()

    documents.append(text)

    doc_ids.append(
        os.path.basename(filepath).replace(".txt", "")
    )

print("Loaded documents:", len(documents))
print("First document:", doc_ids[0])
print(documents[0][:1200])

Loaded documents: 296
First document: DOC-0001
# The Dissolution of Wampa Robotics: A Structural Analysis of Coordinated Corporate Action Within the CGC Framework

**Haven Institute for Policy Studies | Research Paper**
**Dr. Elara Quinshan**
**74-03-12 PCE**
Classification: OPEN

---

## Abstract

The coordinated destruction of Wampa Robotics in 28 PCE remains one of the most consequential acts of collective megacorporate force in the Post-Cascade Era. Executed by the combined forces of the Original 5 Firms at a ceremony commemorating the 25th anniversary of the Clairos Governance Council, the event inaugurated a new phase of inter-megacorp relations defined by mutual suspicion, formalized non-aggression, and — paradoxically — intensifying structural competition. This paper examines the governance mechanics that enabled the coordinated action, the immediate aftermath in terms of robotics industry structure, and the medium-term consequences for CGC decision-making capacity. It argues t

Recursive Chunk

create chunk

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = []
chunk_to_doc = []

for doc_id, document in zip(doc_ids, documents):

    split_chunks = text_splitter.split_text(
        document
    )

    for chunk in split_chunks:

        chunks.append(chunk)

        chunk_to_doc.append(doc_id)

print("Total chunks:", len(chunks))
print()
print(chunks[0][:500])

Total chunks: 6058

# The Dissolution of Wampa Robotics: A Structural Analysis of Coordinated Corporate Action Within the CGC Framework

**Haven Institute for Policy Studies | Research Paper**
**Dr. Elara Quinshan**
**74-03-12 PCE**
Classification: OPEN

---

## Abstract


Load Embedding Model and Create Embedding

In [5]:
embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/190 [00:00<?, ?it/s]

Build FAISS Index

In [7]:
dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    dimension
)

faiss_index.add(
    np.array(chunk_embeddings).astype("float32")
)

print("FAISS ready.")

FAISS ready.


Build BM25

In [8]:
tokenized_chunks = [
    chunk.lower().split()
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 ready!")

BM25 ready!


Test BM25

In [9]:
query = "How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?"

tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)

top_indices = np.argsort(scores)[::-1][:5]

for idx in top_indices:

    print("="*80)

    print("DOC:", chunk_to_doc[idx])

    print("BM25 SCORE:", scores[idx])

    print()

    print(chunks[idx][:500])

DOC: DOC-0255
BM25 SCORE: 72.31106786337813

Phase II of the Sharpsea Route Optimization Program is operational. Following your personal review and approval of Phase I in 75 PCE, this office was directed to extend algorithmic routing to the full working Sharpsea fleet. As of 76-10-30 PCE, the new algorithmic routing suite has been deployed across 74 ONE vessels operating in Sharpsea Bloc shipping lanes. Deployment was completed on schedule and within allocated operational budget
DOC: DOC-0255
BM25 SCORE: 35.651591025191955

---

**FLEET DISPOSITION**

ONE Network currently maintains 31 vessels on permanent Sharpsea Bloc rotation. The Phase II deployment encompassed the broader operational fleet cycling through the region, which exceeds our permanent rotation complement. Fleet commanders have reported no significant mechanical or integration issues with the new routing suite. Vessel positioning is being managed in accordance with existing transit agreements.

---

**DIPLOMATIC CONCERNS*

Hybrid Retrivel

In [10]:
def hybrid_search(question, top_k=30, alpha=0.7, dense_top_n=50, sparse_top_n=50):
    # Dense retrieval (top dense_top_n)
    q_emb = embedding_model.encode([question], normalize_embeddings=True)
    D, I = faiss_index.search(np.array(q_emb).astype("float32"), k=dense_top_n)

    dense_scores = {int(idx): float(score) for score, idx in zip(D[0], I[0]) if idx >= 0}

    # Sparse retrieval (top sparse_top_n)
    tokenized_query = question.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    top_sparse_idx = np.argpartition(bm25_scores, -sparse_top_n)[-sparse_top_n:]
    sparse_scores = {int(i): float(bm25_scores[i]) for i in top_sparse_idx}

    candidates = set(dense_scores.keys()) | set(sparse_scores.keys())

    # Normalize sparse over candidates only
    sparse_vals = np.array([sparse_scores.get(i, 0.0) for i in candidates], dtype="float32")
    s_min, s_max = float(sparse_vals.min()), float(sparse_vals.max())
    def norm_sparse(v): return (v - s_min) / (s_max - s_min + 1e-9)

    combined = []
    for idx in candidates:
        dense = dense_scores.get(idx, 0.0)
        sparse = norm_sparse(sparse_scores.get(idx, 0.0))
        score = alpha * dense + (1 - alpha) * sparse
        combined.append((idx, float(score)))

    combined.sort(key=lambda x: x[1], reverse=True)
    combined = combined[:top_k]

    results = []
    for idx, score in combined:
        results.append({
            "doc_id": chunk_to_doc[idx],
            "chunk": chunks[idx],
            "score": score,
            "chunk_index": idx
        })
    return results

Test Retrivel

In [11]:
results = hybrid_search(
    "How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?"
)

for r in results:

    print("="*80)

    print("DOC:", r["doc_id"])

    print("SCORE:", r["score"])

    print()

    print(r["chunk"][:500])

DOC: DOC-0255
SCORE: 0.916631570597446

Phase II of the Sharpsea Route Optimization Program is operational. Following your personal review and approval of Phase I in 75 PCE, this office was directed to extend algorithmic routing to the full working Sharpsea fleet. As of 76-10-30 PCE, the new algorithmic routing suite has been deployed across 74 ONE vessels operating in Sharpsea Bloc shipping lanes. Deployment was completed on schedule and within allocated operational budget
DOC: DOC-0255
SCORE: 0.6869805250879246

---

**FLEET DISPOSITION**

ONE Network currently maintains 31 vessels on permanent Sharpsea Bloc rotation. The Phase II deployment encompassed the broader operational fleet cycling through the region, which exceeds our permanent rotation complement. Fleet commanders have reported no significant mechanical or integration issues with the new routing suite. Vessel positioning is being managed in accordance with existing transit agreements.

---

**DIPLOMATIC CONCERNS**
DOC: DOC

# Load Reranker

In [12]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

print("Reranker loaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranker loaded


In [13]:
# Reranking Function

In [14]:
def rerank_results(question, retrieved_results, top_k=5):

    pairs = []

    for result in retrieved_results:

        pairs.append([
            question,
            result["chunk"]
        ])

    rerank_scores = reranker.predict(pairs)

    reranked = []

    for i in range(len(retrieved_results)):

        reranked.append({
            "doc_id": retrieved_results[i]["doc_id"],
            "chunk": retrieved_results[i]["chunk"],
            "score": rerank_scores[i]
        })

    reranked = sorted(
        reranked,
        key=lambda x: x["score"],
        reverse=True
    )

    return reranked[:top_k]

In [15]:
# Test Reranking

In [16]:
query = "How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?"

retrieved = hybrid_search(query)

reranked = rerank_results(
    query,
    retrieved,
    top_k=5
)

for r in reranked:

    print("="*80)

    print("DOC:", r["doc_id"])
    print("RERANK SCORE:", r["score"])

    print()
    print(r["chunk"][:500])

    print()

DOC: DOC-0255
RERANK SCORE: 0.9999666

Phase II of the Sharpsea Route Optimization Program is operational. Following your personal review and approval of Phase I in 75 PCE, this office was directed to extend algorithmic routing to the full working Sharpsea fleet. As of 76-10-30 PCE, the new algorithmic routing suite has been deployed across 74 ONE vessels operating in Sharpsea Bloc shipping lanes. Deployment was completed on schedule and within allocated operational budget

DOC: DOC-0255
RERANK SCORE: 0.9994281

---

**FLEET DISPOSITION**

ONE Network currently maintains 31 vessels on permanent Sharpsea Bloc rotation. The Phase II deployment encompassed the broader operational fleet cycling through the region, which exceeds our permanent rotation complement. Fleet commanders have reported no significant mechanical or integration issues with the new routing suite. Vessel positioning is being managed in accordance with existing transit agreements.

---

**DIPLOMATIC CONCERNS**

DOC: DOC-

Why reranking matters

Right now:

FAISS retrieves by embedding similarity
BM25 retrieves by keyword overlap

But BOTH are imperfect.

Reranker:

deeply reads the chunk
compares question ↔ chunk together
produces much more accurate ranking

This is usually one of the biggest RAG improvements.

In [17]:
#Answer Generator

In [18]:
!pip install -q transformers accelerate bitsandbytes

In [19]:
# Load Qwen2.5-7B-Instruct

In [20]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Qwen loaded")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen loaded


In [21]:
# Build RAG Prompt

In [22]:
def build_prompt(question, contexts):

    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a factual retrieval-based QA assistant.

Use ONLY the provided context.

You may combine information across multiple passages.

If the answer is implied by the context,
provide the best concise answer.

If the answer truly cannot be determined,
reply EXACTLY:
NOT_FOUND

Context:
{context_text}

Question:
{question}

Answer:
"""

    return prompt

In [23]:
# Generate Answer

In [24]:
def generate_answer(question):

    retrieved = hybrid_search(question)

    reranked = rerank_results(
        question,
        retrieved,
        top_k=3
    )

    contexts = [
        r["chunk"]
        for r in reranked
    ]

    prompt = build_prompt(
        question,
        contexts
    )

    output = generator(
        prompt,
        max_new_tokens=80,
        do_sample=False
    )

    generated_text = output[0]["generated_text"]

    answer = generated_text.split("Answer:")[-1].strip()

    return {
        "question": question,
        "answer": answer,
        "documents": [
            r["doc_id"]
            for r in reranked
        ]
    }

In [25]:
# Test Full RAG Pipeline

In [26]:
result = generate_answer(
    "How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nDOCUMENTS:")
print(result["documents"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


QUESTION:
How many ONE vessels had the new algorithmic routing deployed across Sharpsea Bloc shipping lanes under Phase II of the Sharpsea Route Optimization Program?

ANSWER:
74

DOCUMENTS:
['DOC-0255', 'DOC-0255', 'DOC-0161']


In [27]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared")

GPU cache cleared
